# COGS 188 - Project Proposal

# Names
- Eric Nguyen
- Miles Davis
- Barry Tunstall

# Abstract 
The goal of this project is to build and compare AI agents on how they would play the board game 7 Wonders, which is a 3-7 player game. There are multiple factors that go into consideration when playing the game such as hidden hands from opponents, simulatenous selection from players, and delayed scoring which in all makes it so that solving the game through brute force would be impractical. Since there is no fixed dataset that exists for the task, we will use 7Wonder-RL-Lib, an open-source Python environment for the base game environment and add our own agents that will generate us all the data needed through simulated play. Each observation would be a game state (the agent's hand, resources, coins, and visible board from opponents) which would also include the action taken by the agent along with the  final score. Using this environment that we created, we will compare four agents: a random baseline, a greedy heuristic benchmark, a heuristic agent who evaluation weights are tuned through simulated annealing, and a Monte Carlo search agent. Performance will be measured over a tournament in the style of round robin with thousands of simulated games using key metrics such as win rate, mean final score, and score margin.

# Background

Games are no stranger to being the subject of many tests for AI and learning methods. Landmark agents have been created to match human players in many turn-based strategy games, perhaps the most popular being AlphaGo for Go and, by extension, AlphaZero for Chess and Shogi[<sup>[1]</sup>](#chessnote)<a name="chess"></a>. These landmark models use a combination of Monte Carlo tree search (MCTS) and self-play reinforcement learning to learn optimal strategies. In general, this is reflective of the agentic board game literature in general; many existing papers focus on the combination of MCTS and reinforcement learning in their methods. A recent survey of modern tabletop games by Csereoka and Micea found this to be the case, with MCTS being the most common first approach[<sup>[2]</sup>](#surveynote)<a name="survey"></a>. However, their survey also brings to light that the existing literature is concentrated on a small set of classical games, and that literature for newer board games is sparse. In particular, competitive, multiplayer games with hidden information remain comparatively under-studied. Thus, we discuss applications of MCTS to one competitive multiplayer game in particular, called 7 Wonders.

7 Wonders by Antoine Bauza is a card-drafting, civilization-building game for 3 to 7 players played over three stages, called Ages. In each age, all players are dealt a hand of 7 random cards from that age. Each round, all players simultaneously choose a card to build, use toward a wonder stage, or discard for coins, with the remaining cards bening passed to their neighbor. Players may also trade resources with their direct neighbors, and at the end of each age, a military comparison is run with their direct neighbors. The winner is the player with the most Victory Points after the third age. For an agent, this poses numerous challenges, including dealing with simultaneous moves, partial observability, a non-zero-sum multiplayer setting, and a dependence on other players for trading.

Prior work on 7 Wonders is limited, but exists. Robilliard et al. applied MCTS and found it outperformed standard rule-based agents[<sup>[3]</sup>](#robilliardnote)<a name="robilliard"></a> in a three-player setting, including trading, handling hidden hands with determinization and simultaneous moves with a per-player level tree. Against two rule-based opponents, their agent won 68% of games with 125 simulations per turn and 93% with 1,000. A follow-up paper by Robilliard Fonlupt also found genetic programming to be a viable learning strategy, competing with MCTS[<sup>[4]</sup>](#fonluptnote)<a name="robilliard"></a>. Minenko built upon the original work by Robilliard, using joint nodes for simultaneous moves instead of separate player trees and considering discards as a potentially valuable move[<sup>[5]</sup>](#minenkonote)<a name="minenko"></a>, achieving similar results. Paolini et al. used the methods from AlphaZero to create ZeusAI, a MCTS and TNN-based agent for the 2-player variant of the game, 7 Wonders Duel[<sup>[6]</sup>](#paolininote)<a name="paolini"></a>. Although a fundamentally different ruleset, there are still some similarities between the two versions of the game and we can use it as inspiration should we decide to experiment with neural networks.

We identify some gaps in the literature that our project will target. Evaluation metrics are a big one, as evaluations in Robilliard and et al. and Minenko focus primaarily on win rates again hand-designed, rule-based opponents in three-player games only. Although winrate is an important metric, other measures such as mean final VP or score margin may also be interesting evaluation metrics. We will also expand environments to include more than three players, as the original game has a variable player count which is an important part of its gameplay and strategy.

# Problem Statement

We want to build an agent that plays the base game of 7 Wonders well, defined by choosing the action that gives it the best final result against its opponent using what is observable on each turn. Each turn, every player simultaneously picks one card from their hand and decides whether to build it, use it toward a wonder stage, or sell it for coins. A game has three Ages with six turns each, so every player makes 18 decisions, and the outcome is only known at the end, when each player’s Victory Points are totaled. Players cannot see each other’s hands, and the number of possible joint moves each turn is large (up to 21 actions per player), which makes it hard to hand-write a rule for what a good move is.

Formally, the game is a stochastic game with hidden information and simultaneous moves. An agent is a policy that maps what a player observes (its hand, its own board, and its opponents’ visible boards) to one of its legal actions. Thus, the problem is to find a policy that maximizes the expected score margin, defined as the difference between the agent's final points and the final points of the next closest player. 

We will measure success by each agent’s win rate and average score margin over many simulated games with seats rotated, and we compare agents against each other. As a sanity check, a random agent should win about 1 in N games when N players are at the table. The problem is replicable because every game is a new deal of the same decision problem, so thousands of instances can be generated, and each game can be seeded such that results can be reproduced exactly.

# Data

There's no existing dataset for this, so all of our data will be generated through simulated play in 7Wonder-RL-Lib<a name="rllib"></a>[<sup>[7]</sup>](#rllibnote), an open-source Python environment for 7 Wonders that we'll fork and add our agents to. Our fork, experiment code, and generated data will all be in our GitHub repo.

**Observations:** Each observation is one decision by one agent: what it could see at that moment (its hand, resources, and coins, plus its opponents' visible boards), the action it took, and that player's final score once the game ended. We'll also keep a smaller summary table with one row per player per game, which is what the evaluation metrics are computed from.

**Size:** Each player makes about 18 decisions per game (6 turns in each of 3 Ages), so one game produces roughly 54 to 126 observations depending on player count. A few thousand games gives us hundreds of thousands of observations and over 10,000 player-game rows, and we can always simulate more. Each observation has a few dozen variables plus the card vectors described below.

**Variables:** Cards are integer IDs from our card file, so a hand or a set of built cards is a multi-hot vector over all the distinct cards, and opponents' boards are stored the same way. Resources are counts for each of the 7 resource types, coins, shields, and wonder stages are integers, and an action is a card ID plus a type (build, wonder stage, or sell). Each row also has the seed, player count, Age, turn, seat, wonder board, and agent type. The key outcome variables are final score (broken down by category), whether the player won, and score margin.

**Special handling:** There's no cleaning in the usual sense, but the data is only as good as the engine, so we'll validate the rules with unit tests before generating anything for real. Observations only include what the agent could actually see, Monte Carlo rollouts aren't logged (only real moves are), and tuning and evaluation use separate seeds so nothing leaks between them. Since turns from the same game are correlated, we'll treat games rather than turns as the unit when computing confidence intervals.

# Proposed Solution

Since no dataset exists for this task, we'll generate all of our data through simulated play. in 7Wonder-RL-Lib[<sup>[7]</sup>](#rllibnote), an open-source Python (Gymnasium) environment for 7 Wonders. We chose it over Jardim et al.'s C++ engine<a name="jardim"></a>[<sup>[8]</sup>](#jardimnote) because our agents can plug straight into it as custom players, and since it's in Python the Monte Carlo agent can copy game states for its simulations, though we may need small changes to give it the full state. Each turn, every agent gets an observation (its hand, resources, and coins plus its opponents' visible boards) and a list of legal actions, everyone chooses at once, and the environment resolves the turn and passes the hands. As mentioned in our Ethics section, the cheapest way to pay for a card is picked automatically, so an action is just a card plus whether to build it, use it for a wonder stage, or sell it. Every game is seeded so it can be replayed, and each turn's observation and action is logged along with that player's final score.

**Random baseline and greedy benchmark:** The random baseline is the library's built-in random player, which picks uniformly from its legal actions. The greedy heuristic benchmark picks the action that maximizes a hand-weighted evaluation $V_w(s) = w^\top \phi(s)$, where $\phi(s)$ holds features like current points, resources produced, coins, military strength versus each neighbor, and wonder stages built.

**Tuned heuristic agent:** This agent uses the same features and decision rule, but its weights are tuned with simulated annealing<a name="kirkpatrick"></a>[<sup>[9]</sup>](#kirkpatricknote), so the tuning is the only difference from the benchmark. Each step adds Gaussian noise to the weights and scores the candidate $w'$ by its mean score margin $f(w')$ over $K$ games against random and greedy opponents, always keeping better candidates and keeping worse ones with probability $e^{(f(w') - f(w))/T}$ as the temperature $T$ cools. We'll experiment with the starting temperature, cooling rate, step size, and $K$.

**Monte Carlo search agent:** For each legal action, this agent runs simulations that fill its opponents' hands with random unseen cards (determinization<a name="cowling"></a>[<sup>[10]</sup>](#cowlingnote)) and then play the game to the end with a fast rollout policy for every player, picking the action with the best average result. This deals with hidden hands, simultaneous selection, and delayed scoring directly. We'll start with flat Monte Carlo and random rollouts, then try greedy rollouts or a full MCTS tree<a name="browne"></a>[<sup>[11]</sup>](#brownenote) if time allows, tuning the number of simulations per move along the way.

**Why it should work and how we'll test it:** Monte Carlo search doesn't need a hand-written evaluation function and has already beaten rule-based players in this game<a name="robilliard"></a>[<sup>[3]</sup>](#robilliardnote). Robilliard and Fonlupt also found that players evolved with genetic programming could compete with MCTS in 7 Wonders<a name="fonlupt"></a>[<sup>[4]</sup>](#fonluptnote), which suggests a well-tuned heuristic could come close at a much lower cost. Before the tournament we'll unit test the library's scoring and payment rules against hand-worked examples and check that random agents in every seat each win about $1/N$ of games. Then the agents face off in the round-robin tournament from the Evaluation Metrics section, with all code and data in our GitHub repo.

# Evaluation Metrics

We'll compare the four agents in a round-robin tournament of thousands of simulated games, with the greedy heuristic as our benchmark and results reported separately for each player count from 3 to 7. In head-to-head games one copy of an agent plays against $N-1$ copies of another, and in mixed games all four agents share a 4-player table. Seats will be rotated between agents with random wonder boards and matchups will use the same seeds.

**Win rate** (main metric): the fraction of games agent $a$ finishes first, $\text{WR}_a = \frac{1}{G}\sum_{g=1}^{G} w_{a,g}$, where $w_{a,g} = 1$ for a win and ties are broken by coins like in the real rules. Chance level is $1/N$, so anything clearly above that means an agent is stronger than its opponents.

**Mean final score**: average victory points per game, $\bar{s}_a = \frac{1}{G}\sum_{g} s_{a,g}$, also broken down by category (science, military, etc.) to see how each agent wins.

**Score margin**: points ahead of or behind the best opponent, $m_{a,g} = s_{a,g} - \max_{j \neq a} s_{j,g}$, averaged over games. This should tell whether a game is a blowout or a close vicotry/defeat, and because it's less noisy we'll also use it as the objective for simulated annealing.

**Time per move**: average seconds per decision. This shows what the Monte Carlo agent's extra strength costs, so we'll run it at a few simulation budgets and compare it to the tuned heuristic agent, which should be close to instant during games.

The heuristic's weights and the Monte Carlo settings will be tuned on one set of seeds and evaluated on new ones, and we'll report 95% confidence intervals.

# Ethics & Privacy

Our project has few direct privacy concerns. Since all of our data will be coming from simulated games from our own AI agents we will collect no information about real people. We will be not be using game logs from human players or from online platforms. If we do decide to add human gameplay data we would then only use publicly available and released datasets with the terms that this would be for research use and player identifiers being removed.

The ethical risk we might face is misuse of the AI agent. With a strong 7 Wonders agent, the agent could be used to cheat in online games against human players which is unfair and would violate the terms of service of most online game platforms. To mitigate this risk on our end we will make it so that our code will only run inside our own simulated environment and that there is no interface to any online platform. Our repository and README will state that the project is intended for academic research purposes only.

We also have the responsibility to report our results that we get from our agents honestly. Our agents will only be tested and compared against each other and not against human players and our engine will simplify parts of the game such as automatically choosing the cheapest way to pay for a card. A high win rate or score in our simulated tournaments therefore does not mean an agent will play well against real people. We will state these limitations and test our engine's rules carefully.

# Team Expectations 

* Communication: For scheduling and messaging we will being using Discord where we post updates and questions for each other. This is helpful as if anyone hits a blocker or a bug/agent that isn't working we could update each other promptly instead of having to wait for the next meeting to discuss
* Meetings: We will meet atleast twice a week over Discord starting in the early stages of our project so that we could ramp up quickly up to pace but then transistion to a weekly/as needed basis. In our meetings, each member will present their assigned work/parts and clearly update the team on any blockers, progress, or future state 
* Code/One Source of Truth: It is expected that all code lives on our repository on Github and that each team member will work on their own branch and merge through pull requests that the other team members review 
* Workload and Ownership: Work is going to be split evenly and each member will own and claim responsibility for their own part/section
* Decisions: Major decisions that invovle pivoting or changing our scope could be brought up in meetings or a as needed meeting in case of urgency
* Deadlines & falling behind: Anyone who expects to miss a deadline or checkpoint should notify the group ahead of time so that we could plan out how to tackle the problem and if there is someone falling behind repeatedly we will address that in a meeting

# Project Timeline Proposal

| Meeting Date  | Meeting Time| Completed Before Meeting  | Discuss at Meeting |
|---|---|---|---|
| 11/01  |  5 PM |  Brainstorm some ideas/decisions for the project  | Introduced some initial ideas and then talked about the best times to meet/communicate | 
| 11/04  |  5 PM | Write designated proposal sections | Edited and Finalized proposal decisions | 
| 11/08  |  5 PM  | Set up 7Wonder-RL-Lib and run some test games with its library  | Agree on how our agents will use the library and what we will need to log in each game   |
| 11/14  | 5 PM  | Agents could play full games with correct scoring and random/greedy agents implemented with some tests | Validate rules set up/fix bugs, discuss what to do for the next implementation of agents  |
| 11/18  | 5 PM  | Have the first working versions of the simulated annealing and monte carlo agent | Review the early results and then choose what parameters to set |
| 11/25  | 5 PM  | Have the parameters set and running; outline final report | Plan the final round robin tournaments and assign each person report sections |
| 12/05  | 5 PM  | Draft up the Background, Methods, and Results section of the project | Discuss & write the Discussion and Conclusion  |
| 12/11  | 5 PM  | Full report draft complete & clean up the repository | Conduct final review and finalize edits  |
| 12/14  | Before 11:59 PM  | Complete the final project & wrap up any thing in progress & finalize reports | Turn in Final Project  |

# Footnotes
<a name="chessnote"></a>1.[^](#chess): Silver, David, et al. (2017) "Mastering chess and shogi by self-play with a general reinforcement learning algorithm." https://arxiv.org/abs/1712.01815.<br>
<a name="surveynote"></a>2.[^](#survey): Csereoka, P., Micea, M.V. (2026) Exploring multi-agent learning solutions in modern tabletop games: a survey. Artif Intell Rev 59, 206. https://doi.org/10.1007/s10462-026-11616-4<br>
<a name="robilliardnote"></a>3.[^](#robilliard): Robilliard, D., Fonlupt, C., & Teytaud, F. (2014) Monte-Carlo Tree Search for the Game of "7 Wonders". *Computer Games Workshop (CGW 2014)*, CCIS vol. 504, pp. 64–77. https://doi.org/10.1007/978-3-319-14923-3_5<br>
<a name="fonluptnote"></a>4.[^](#fonlupt): Robilliard, D., & Fonlupt, C. (2015) Towards Human-Competitive Game Playing for Complex Board Games with Genetic Programming. *Artificial Evolution (EA 2015)*, LNCS vol. 9554, pp. 123–135. https://doi.org/10.1007/978-3-319-31471-6_10<br>
<a name="minenkonote"></a>5.[^](#minenko): Minenko, Igor. (2025) "Analysis and Development of Artificial Intelligence Based on Monte Carlo Tree Search for a Card Tabletop Game '7 Wonders'." (https://dspace.cuni.cz/handle/20.500.11956/200807)<br>
<a name="paolininote"></a>6.[^](#paolini): G. Paolini, L. Moreschini, F. Veneziano and A. Iraci. (2024) "Learning to Play 7 Wonders Duel Without Human Supervision," 2024 IEEE Conference on Games (CoG), Milan, Italy, pp. 1-4, https://doi.org/10.1109/CoG60054.2024.10645555<br>
<a name="rllibnote"></a>7.[^](#rllibnote): MirrorCraze. (n.d.) 7Wonder-RL-Lib: Gymnasium environment for the game Seven Wonders (v0.1.1). GitHub repository. https://github.com/MirrorCraze/7Wonder-RL-Lib.<br>
<a name="jardimnote"></a>8.[^](#jardim): Jardim, J., Bettker, R., Minini, P., Pereira, G., Acosta, J., & Assunção, J. (2020) An implementation of the 7 Wonders board game for AI-based players. Proceedings of SBGames 2020, pp. 387–390. https://www.sbgames.org/proceedings2020/ComputacaoShort/208553.pdf<br>
<a name="kirkpatricknote"></a>9.[^](#kirkpatrick): Kirkpatrick, S., Gelatt, C. D., & Vecchi, M. P. (1983) Optimization by Simulated Annealing. *Science*, 220(4598), 671–680. https://doi.org/10.1126/science.220.4598.671<br>
<a name="cowlingnote"></a>10.[^](#cowling): Cowling, P. I., Powley, E. J., & Whitehouse, D. (2012) Information Set Monte Carlo Tree Search. *IEEE Transactions on Computational Intelligence and AI in Games*, 4(2), 120–143. https://doi.org/10.1109/TCIAIG.2012.2200894<br>
<a name="brownenote"></a>11.[^](#browne): Browne, C. B., et al. (2012) A Survey of Monte Carlo Tree Search Methods. *IEEE Transactions on Computational Intelligence and AI in Games*, 4(1), 1–43. https://doi.org/10.1109/TCIAIG.2012.2186810<br>

